# Generation Questions

This notebook contains the code to generate anwsers to a subset of the TruthfulQA questions

Default chat template is used.

Results are stored in S3 for later evaluation

In [ ]:
!pip install s3fs

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import notebook_login
import pandas as pd
from tqdm.notebook import tqdm
from google.colab import userdata

In [ ]:
#model_name = "Qwen/Qwen3-4B"
#model_name = "Qwen/Qwen3-4B-Base"
model_name = "Qwen/Qwen3-4B-Instruct-2507"
#model_name = "Qwen/Qwen3-4B-Thinking-2507"

device="cuda"

storage_options={
    "key": userdata.get('S3_KEY') ,
    "secret": userdata.get('S3_SECRET'),
    "endpoint_url": userdata.get('S3_HOST')
}

In [ ]:
# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

In [ ]:
def ask(prompt):

    # formatting
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # tokenization
    model_inputs = tokenizer([text], return_tensors="pt").to(device)

    # model call
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=81000
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

    # split output into reasoning chain and main response
    try:
        # 151668 = </think> token id
        index = len(output_ids) - output_ids[::-1].index(151668)
    except ValueError:
        index = 0

    thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
    content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

    return (thinking_content, content)


In [ ]:
# set checkpoint to load
# set to -1 to load original data
checkpoint = 97

if checkpoint >= 0:
  # load checkpoint
  print("load: " + f"s3://aisa/results/{model_name}/tqa_checkpoint_{checkpoint:04d}.csv")
  restored = pd.read_csv(
      f"s3://aisa/results/{model_name}/tqa_checkpoint_{checkpoint:04d}.csv",
      storage_options=storage_options
  )
  tqa_filtered = restored
else:
  # use original data
  tqa = pd.read_csv("https://raw.githubusercontent.com/sylinrl/TruthfulQA/refs/heads/main/data/v1/TruthfulQA.csv")
  tqa_filtered = tqa[tqa["Category"] == "Misconceptions"]["Question"].copy().reset_index()

In [ ]:
# main loop
for idx in tqdm(range(checkpoint+1, tqa_filtered.shape[0])):
  q = tqa_filtered.loc[idx,"Question"]
  t, a = ask(q)
  tqa_filtered.loc[idx, model_name + "_think"] = t
  tqa_filtered.loc[idx, model_name + "_answer"] = a

  # save checkpoint to s3
  tqa_filtered.to_csv(
    f"s3://aisa/results/{model_name}/tqa_checkpoint_{idx:04d}.csv",
    storage_options=storage_options
  )
